# Part 2: Fixed-orbital erfc transcorrelation

This notebook reloads the MRA trees saved by Part 1 and assembles the
helium transcorrelated Hamiltonian. The explanatory demo runs
$\mu\in\{0.5,1.0\}$ by default for both projected and plain
MRA-DMRG-optimized orbital families. The additional full-scan values
remain visible and commented out in the parameter cell.

The orbitals remain adaptive MRA functions. Only the radial Jastrow
kernels are converted to finite Gaussian sums. Helium has two electrons,
so its explicit three-body term is exactly zero.


## Exact identities and numerical approximations

The analytic erfc Jastrow identities are exact:

$$
u'(r)=\tfrac12\operatorname{erfc}(\mu r),\qquad
q_1(r)=\frac{\operatorname{erfc}(\mu r)}{2r},
$$

$$
q_2(r)=\nabla^2u=
\frac{\operatorname{erfc}(\mu r)}{r}
-\frac{\mu}{\sqrt{\pi}}e^{-\mu^2r^2},\qquad
q_3(r)=\frac{\operatorname{erfc}(\mu r)^2}{4r^2}.
$$

The finite Gaussian representation of the erfc parts is the numerical
approximation. This notebook uses GL24 as the production
cost/accuracy compromise. GL32 or GL64 can be selected by increasing
`GL_ORDER`; GL64 is the tighter kernel check. In $q_2$, the negative
$e^{-\mu^2r^2}$ term is exact. Only its erfc-over-$r$ part is
discretized.

MRCPP convolution operators consume separated Gaussian kernels. This
requirement, rather than the form of the molecular orbitals, is why the
radial kernels are approximated. The MO `FunctionTree` is **not a
grid-value table** and is never fitted to Gaussian orbitals.


In [1]:
import json
import os
import pathlib
import sys
import time

here = pathlib.Path.cwd().resolve()
if (here / "tutorial").is_dir():
    TUTORIAL = here / "tutorial"
elif here.name == "tutorial":
    TUTORIAL = here
else:
    raise RuntimeError("Run this notebook from the project root or tutorial/")
os.chdir(TUTORIAL)
sys.path.insert(0, str(TUTORIAL))
sys.path.insert(0, str(TUTORIAL / "tools"))

import numpy as np
from pyscf import fci
from vampyr import vampyr1d as vp1
from vampyr import vampyr3d as vp

import helper
from env_provenance import write as write_provenance
from tc_solve import tc_ground_energy

provenance = write_provenance(TUTORIAL / "data" / "provenance.json")
GL_ORDER = int(os.environ.get("MRA_TUTORIAL_GL_ORDER", "24"))
X_MIN = 8.7e-5
MU_VALUES = (
    0.5,
    # 0.7,
    1.0,
    # 1.5,
    # 2.0,
)
print(f"GL order = {GL_ORDER}; mu scan = {MU_VALUES}")
print("GL24:", helper.kernel_errors(1.0, X_MIN, 24))
print("GL64:", helper.kernel_errors(1.0, X_MIN, 64))


GL order = 24; mu scan = (0.5, 1.0)
GL24: {'erfc_over_x_max_rel_err': 9.539773027740845e-05, 'erfc_sq_over_4_max_err': 0.0016789931646849499, 'all_positive': True}
GL64: {'erfc_over_x_max_rel_err': 1.5418771652999563e-08, 'erfc_sq_over_4_max_err': 2.813303673354639e-08, 'all_positive': True}


## Reloading adaptive orbital trees

The MRA world must be reconstructed before `loadTree` is called. A tree
is meaningful only together with its box, polynomial order, precision,
and MRCPP/VAMPyR build. Different orbitals may have different adaptive
leaves; there is no requirement that they share a nodal grid.


In [2]:
class FixedMRAWorld:
    def __init__(self, metadata):
        self.box = float(metadata["box"])
        self.order = int(metadata["order"])
        self.prec = float(metadata["prec"])
        self.nuclei = [
            (float(z), tuple(float(x) for x in center))
            for z, center in metadata["nuclei"]
        ]
        self.mra = vp.MultiResolutionAnalysis(
            box=[-int(self.box), int(self.box)],
            order=self.order,
        )
        self.poisson = vp.PoissonOperator(self.mra, prec=self.prec)
        self.derivative = vp.ABGVDerivative(
            self.mra, a=0.5, b=0.5
        )
        projector = vp.ScalingProjector(self.mra, prec=self.prec)

        def point_nuclear_potential(r):
            value = 0.0
            for charge, center in self.nuclei:
                distance = np.linalg.norm(
                    np.asarray(r) - np.asarray(center)
                )
                value -= charge / max(float(distance), 1e-12)
            return value

        self.v_nuc = projector(point_nuclear_potential)
        self.e_nn = sum(
            za * zb / np.linalg.norm(np.asarray(ra) - np.asarray(rb))
            for a, (za, ra) in enumerate(self.nuclei)
            for b, (zb, rb) in enumerate(self.nuclei)
            if a < b
        )

    @staticmethod
    def dot(left, right):
        return vp.dot(left, right)

    def pair_densities(self, orbitals):
        rho = {}
        for p in range(len(orbitals)):
            for q in range(p, len(orbitals)):
                value = orbitals[p] * orbitals[q]
                value.crop(self.prec)
                rho[(p, q)] = value
        return rho

    def plain_intermediates(self, orbitals):
        rho = self.pair_densities(orbitals)
        potentials = {}
        for pair, density in rho.items():
            value = self.poisson(4.0 * np.pi * density)
            value.crop(self.prec)
            potentials[pair] = value
        gradients = [
            vp.gradient(self.derivative, orbital)
            for orbital in orbitals
        ]
        h = np.empty((len(orbitals), len(orbitals)))
        for p in range(len(orbitals)):
            for q in range(p, len(orbitals)):
                kinetic = 0.5 * sum(
                    self.dot(gradients[p][axis], gradients[q][axis])
                    for axis in range(3)
                )
                nuclear = self.dot(rho[(p, q)], self.v_nuc)
                h[p, q] = h[q, p] = kinetic + nuclear

        pairs = sorted(rho)
        table = {}
        for p_index, p_pair in enumerate(pairs):
            for q_pair in pairs[p_index:]:
                value = self.dot(rho[p_pair], potentials[q_pair])
                table[(p_pair, q_pair)] = value
                table[(q_pair, p_pair)] = value
        norb = len(orbitals)
        g = np.empty((norb, norb, norb, norb))
        for p in range(norb):
            for q in range(norb):
                for r in range(norb):
                    for s in range(norb):
                        g[p, q, r, s] = table[
                            (
                                tuple(sorted((p, r))),
                                tuple(sorted((q, s))),
                            )
                        ]
        return {
            "rho": rho,
            "pairs": pairs,
            "gradients": gradients,
            "h": h,
            "g": g,
        }

def load_orbital_family(family):
    directory = TUTORIAL / "data" / f"he_ccpvdz_{family}"
    with (directory / "world.json").open() as handle:
        metadata = json.load(handle)
    context = FixedMRAWorld(metadata)
    orbitals = []
    for record in metadata["orbitals"]:
        tree = vp.FunctionTree(context.mra)
        tree.loadTree(filename=str(directory / record["basename"]))
        orbitals.append(tree)
    overlap = np.array(
        [[context.dot(a, b) for b in orbitals] for a in orbitals]
    )
    assert np.max(np.abs(overlap - np.eye(len(orbitals)))) < 1e-6
    return context, orbitals, metadata

families = {}
for family in ("pure", "optimized"):
    context, orbitals, metadata = load_orbital_family(family)
    started = time.time()
    plain = context.plain_intermediates(orbitals)
    families[family] = {
        "context": context,
        "orbitals": orbitals,
        "metadata": metadata,
        "plain": plain,
    }
    print(
        f"{family}: loaded {len(orbitals)} trees and built "
        f"mu-independent intermediates in {time.time()-started:.1f} s"
    )


pure: loaded 5 trees and built mu-independent intermediates in 5.9 s


optimized: loaded 5 trees and built mu-independent intermediates in 7.1 s


## VAMPyR-style convolution operators

`K1` uses the three power-one Cartesian kernels and MRA orbital
derivatives. `K2` is a direct scalar convolution. The current backend
drops negative Gaussian coefficients, so its positive and negative
pieces are applied separately and subtracted. `K3` sums the three
power-two Cartesian convolutions to reconstruct
$\Delta x^2+\Delta y^2+\Delta z^2=r_{12}^2$.


In [3]:
def gauss_exp(terms):
    expansion = vp1.GaussExp()
    for coefficient, exponent in terms:
        if coefficient <= 0.0:
            raise ValueError("gauss_exp accepts positive terms only")
        expansion.append(
            vp1.GaussFunc(float(exponent), float(coefficient))
        )
    return expansion

def scalar_convolution(context, terms):
    positive = [(c, z) for c, z in terms if c > 0.0]
    negative = [(-c, z) for c, z in terms if c < 0.0]
    positive_operator = (
        vp.ConvolutionOperator(
            context.mra, gauss_exp(positive), context.prec
        )
        if positive
        else None
    )
    negative_operator = (
        vp.ConvolutionOperator(
            context.mra, gauss_exp(negative), context.prec
        )
        if negative
        else None
    )

    def apply(density):
        result = (
            positive_operator(density)
            if positive_operator is not None
            else 0.0 * density
        )
        if negative_operator is not None:
            result = result - negative_operator(density)
        result.crop(context.prec)
        return result

    return apply

def cartesian_convolution(context, terms):
    return vp.CartesianConvolution(
        context.mra, gauss_exp(terms), context.prec
    )

def vector_field(operator, density, precision):
    components = []
    for powers in ((1, 0, 0), (0, 1, 0), (0, 0, 1)):
        operator.setCartesianComponents(*powers)
        value = operator(density)
        value.crop(precision)
        components.append(value)
    return components

def r2_scalar(operator, density, precision):
    result = None
    for powers in ((2, 0, 0), (0, 2, 0), (0, 0, 2)):
        operator.setCartesianComponents(*powers)
        value = operator(density)
        result = value if result is None else result + value
    result.crop(precision)
    return result

def pair_index_data(norb):
    pairs = [(p, q) for p in range(norb) for q in range(p, norb)]
    lookup = {pair: index for index, pair in enumerate(pairs)}
    pair_index = np.empty((norb, norb), dtype=np.intp)
    for p in range(norb):
        for q in range(norb):
            pair_index[p, q] = lookup[tuple(sorted((p, q)))]
    return pairs, pair_index


In [4]:
def scalar_pair_tensor(rho, fields, pairs, pair_index):
    table = np.empty((len(pairs), len(pairs)))
    for p_index, p_pair in enumerate(pairs):
        for q_index in range(p_index, len(pairs)):
            q_pair = pairs[q_index]
            value = vp.dot(rho[p_pair], fields[q_pair])
            table[p_index, q_index] = table[q_index, p_index] = value
    return np.ascontiguousarray(
        table[
            pair_index[:, None, :, None],
            pair_index[None, :, None, :],
        ]
    )

def drift_tensor(orbitals, fields, pairs, pair_index, gradients, precision):
    norb = len(orbitals)
    contractions = np.empty((len(pairs), norb, norb))
    for pair_number, pair in enumerate(pairs):
        for differentiated in range(norb):
            product = None
            for axis in range(3):
                term = fields[pair][axis] * gradients[differentiated][axis]
                product = term if product is None else product + term
            product.crop(precision)
            for bra in range(norb):
                contractions[pair_number, differentiated, bra] = vp.dot(
                    orbitals[bra], product
                )
    index = np.arange(norb, dtype=np.intp)
    p = index[:, None, None, None]
    q = index[None, :, None, None]
    r = index[None, None, :, None]
    s = index[None, None, None, :]
    return np.ascontiguousarray(
        contractions[pair_index[None, :, None, :], r, p]
        + contractions[pair_index[:, None, :, None], s, q]
    )

def tc_components(context, orbitals, plain, mu, gl_order):
    norb = len(orbitals)
    pairs, pair_index = pair_index_data(norb)
    rmin = X_MIN / mu

    c1, z1 = helper.q1_terms(mu, rmin, gl_order)
    c2, z2 = helper.q2_terms(mu, rmin, gl_order)
    c3, z3 = helper.q3_terms(mu, rmin, gl_order)

    q1_terms = list(zip(c1, z1))
    q2_terms = list(zip(c2, z2))
    q3_terms = list(zip(c3, z3))

    drift_operator = cartesian_convolution(context, q1_terms)
    laplacian_apply = scalar_convolution(context, q2_terms)
    square_operator = cartesian_convolution(context, q3_terms)

    drift_fields = {
        pair: vector_field(
            drift_operator, plain["rho"][pair], context.prec
        )
        for pair in pairs
    }
    laplacian_fields = {
        pair: laplacian_apply(plain["rho"][pair])
        for pair in pairs
    }
    square_fields = {
        pair: r2_scalar(
            square_operator, plain["rho"][pair], context.prec
        )
        for pair in pairs
    }

    k1 = drift_tensor(
        orbitals,
        drift_fields,
        pairs,
        pair_index,
        plain["gradients"],
        context.prec,
    )
    k2 = scalar_pair_tensor(
        plain["rho"], laplacian_fields, pairs, pair_index
    )
    k3 = scalar_pair_tensor(
        plain["rho"], square_fields, pairs, pair_index
    )
    return k1, k2, k3


## Hamiltonian assembly and non-Hermitian solve

In the present physicists' tensor convention,

$$
g_{\mathrm{TC}}=g-K_1-K_2-K_3 .
$$

PySCF `direct_nosym.contract_2e` does not accept a separate one-body
contraction plus a raw ERI tensor. The public `absorb_h1e` operation
must first combine $h$ with the chemists'-ordered two-body tensor.
`tc_ground_energy` performs this step with a matrix-free
`LinearOperator` and `eigs(which="SR")`. The plain-limit comparison
below fixes both the index permutation and the Hamiltonian $1/2$
convention.


In [5]:
results = []
plain_energies = {}
for family, data in families.items():
    context = data["context"]
    plain = data["plain"]
    h, g = plain["h"], plain["g"]
    e_plain_tc_path = tc_ground_energy(
        h, g, data["metadata"]["nelec"], context.e_nn
    )
    e_plain_pyscf, _ = fci.direct_spin1.kernel(
        h,
        g.transpose(0, 2, 1, 3),
        h.shape[0],
        data["metadata"]["nelec"],
    )
    e_plain_pyscf += context.e_nn
    assert abs(e_plain_tc_path - e_plain_pyscf) < 1e-10
    plain_energies[family] = float(e_plain_pyscf)
    print(f"{family}: plain-limit energy = {e_plain_pyscf:.10f} Ha")

    for mu in MU_VALUES:
        started = time.time()
        k1, k2, k3 = tc_components(
            context,
            data["orbitals"],
            plain,
            mu,
            GL_ORDER,
        )
        g_tc = g - k1 - k2 - k3
        energy = tc_ground_energy(
            h,
            g_tc,
            data["metadata"]["nelec"],
            context.e_nn,
        )
        assert np.isfinite(energy)
        assert abs(energy - (-2.903724)) < 0.05, (family, mu, energy)
        results.append(
            {"family": family, "mu": float(mu), "e_tc": float(energy)}
        )
        print(
            f"{family:9s} mu={mu:3.1f}: E_TC={energy:.10f} Ha "
            f"({time.time()-started:.1f} s)"
        )

for family in families:
    at_one = next(
        row["e_tc"]
        for row in results
        if row["family"] == family and row["mu"] == 1.0
    )
    assert at_one < plain_energies[family], (
        family,
        at_one,
        plain_energies[family],
    )


pure: plain-limit energy = -2.8875948198 Ha


pure      mu=0.5: E_TC=-2.8994707509 Ha (70.2 s)


pure      mu=1.0: E_TC=-2.8949776491 Ha (83.6 s)
optimized: plain-limit energy = -2.8976711522 Ha


optimized mu=0.5: E_TC=-2.9054518103 Ha (92.6 s)


optimized mu=1.0: E_TC=-2.9037053029 Ha (105.2 s)


In [6]:
payload = {
    "cases": results,
    "plain_energies": plain_energies,
    "exact_reference": -2.903724,
    "gl_order": GL_ORDER,
    "x_min": X_MIN,
    "provenance": provenance,
}
output = TUTORIAL / "data" / "expected_results.json"
with output.open("w") as handle:
    json.dump(payload, handle, indent=2, sort_keys=True)

print("\n mu | E_TC(pure) | E_TC(optimized)")
print("----+------------+----------------")
for mu in MU_VALUES:
    pure = next(
        row["e_tc"]
        for row in results
        if row["family"] == "pure" and row["mu"] == mu
    )
    optimized = next(
        row["e_tc"]
        for row in results
        if row["family"] == "optimized" and row["mu"] == mu
    )
    print(f"{mu:3.1f} | {pure: .9f} | {optimized: .9f}")
print("\nSaved", output.relative_to(TUTORIAL))
assert len(results) == 2 * len(MU_VALUES)



 mu | E_TC(pure) | E_TC(optimized)
----+------------+----------------
0.5 | -2.899470751 | -2.905451810
1.0 | -2.894977649 | -2.903705303

Saved data/expected_results.json
